# Gödel trace של forward pass אמיתי

הניסוי הזה **לא מאמן שום מודל** ולא שואל שאלות לוגיות.

הוא טוען מודל pretrained קטן, מעביר משפט אחד, ובונה קוד גדל פורמלי ורקורסיבי של החישוב.

הרעיון:

\[
G(v)=2^{op(v)}3^{G(p_1)}5^{G(p_2)}7^{G(p_3)}\cdots
\]

אנחנו לא מחשבים את המספר הענק; שומרים את הפירוק/העץ שמגדיר אותו.

In [ ]:
!pip -q install transformer_lens

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 1.5 MB/s eta 0:00:00


In [ ]:
import math
from dataclasses import dataclass, field
from typing import List, Dict, Any
import torch
from transformer_lens import HookedTransformer

MODEL_NAME = "pythia-70m"
PROMPT = "The dog chased the red ball."

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
tokens = model.to_tokens(PROMPT)
str_tokens = model.to_str_tokens(PROMPT)

print("device:", device)
print("tokens:", list(enumerate(str_tokens)))

/tmp/ipykernel_590/1741856059.py:11: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)


config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  166MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

AttributeError: 'GPTNeoXForCausalLM' object has no attribute 'embed_out'

In [ ]:
# Forward pass אמיתי אחד. אין training.
with torch.no_grad():
    logits, cache = model.run_with_cache(tokens, return_type="logits")

next_id = int(logits[0, -1].argmax())
print("next-token prediction:", model.to_string(next_id))
print("layers:", model.cfg.n_layers, "| heads:", model.cfg.n_heads, "| sequence length:", tokens.shape[1])

## הקידוד

הקודים של הפעולות נקבעים מראש. הם **לא נלמדים** מהמודל.

In [ ]:
OP = {
    "TOKEN": 1,
    "EMBED": 2,
    "LN1": 3,
    "Q": 4,
    "K": 5,
    "V": 6,
    "ATTN_SCORE": 7,
    "SOFTMAX_ROW": 8,
    "ATTEND": 9,
    "ATTN_OUT": 10,
    "LN2": 11,
    "MLP": 12,
    "ADD": 13,
    "FINAL_LN": 14,
    "UNEMBED": 15,
}

def first_primes(n):
    out = []
    x = 2
    while len(out) < n:
        ok = True
        r = int(x**0.5)
        for p in out:
            if p > r:
                break
            if x % p == 0:
                ok = False
                break
        if ok:
            out.append(x)
        x += 1
    return out

@dataclass
class GNode:
    idx: int
    op: str
    parents: List[int] = field(default_factory=list)
    meta: Dict[str, Any] = field(default_factory=dict)

nodes = []

def add_node(op, parents=None, **meta):
    n = GNode(len(nodes), op, list(parents or []), meta)
    nodes.append(n)
    return n.idx

def formal_code(node_id, max_parents_to_show=6):
    n = nodes[node_id]
    ps = first_primes(1 + len(n.parents))
    parts = [f"2^{OP[n.op]}"]
    for j, pid in enumerate(n.parents[:max_parents_to_show], start=1):
        parts.append(f"{ps[j]}^G{pid}")
    if len(n.parents) > max_parents_to_show:
        parts.append(f"...(+{len(n.parents)-max_parents_to_show} parents)")
    return " * ".join(parts)

In [ ]:
# בונים Gödel computation graph ברמת פעולות-וקטור.
# זו גרסה גסה בכוונה: לא כל כפל סקלרי במטריצה הוא צומת נפרד.

T = tokens.shape[1]
L = model.cfg.n_layers
H = model.cfg.n_heads

token_nodes = []
embed_nodes = []

for t in range(T):
    tok_id = int(tokens[0, t])
    tn = add_node("TOKEN", [], token_id=tok_id, token_text=str_tokens[t], position=t)
    token_nodes.append(tn)
    en = add_node("EMBED", [tn], position=t)
    embed_nodes.append(en)

resid = embed_nodes[:]

for layer in range(L):
    ln1 = []
    q, k, v = {}, {}, {}

    for t in range(T):
        ln1_t = add_node("LN1", [resid[t]], layer=layer, position=t)
        ln1.append(ln1_t)
        for h in range(H):
            q[(h,t)] = add_node("Q", [ln1_t], layer=layer, head=h, position=t)
            k[(h,t)] = add_node("K", [ln1_t], layer=layer, head=h, position=t)
            v[(h,t)] = add_node("V", [ln1_t], layer=layer, head=h, position=t)

    z = {}
    for h in range(H):
        for t in range(T):
            score_nodes = []
            for s in range(t+1):
                sc = add_node(
                    "ATTN_SCORE",
                    [q[(h,t)], k[(h,s)]],
                    layer=layer, head=h, target=t, source=s
                )
                score_nodes.append(sc)

            sm = add_node(
                "SOFTMAX_ROW",
                score_nodes,
                layer=layer, head=h, target=t
            )

            parents = [sm] + [v[(h,s)] for s in range(t+1)]
            z[(h,t)] = add_node(
                "ATTEND",
                parents,
                layer=layer, head=h, target=t
            )

    attn_out = []
    for t in range(T):
        ao = add_node(
            "ATTN_OUT",
            [z[(h,t)] for h in range(H)],
            layer=layer, position=t
        )
        attn_out.append(ao)

    new_resid = []
    for t in range(T):
        ln2 = add_node("LN2", [resid[t]], layer=layer, position=t)
        mlp = add_node("MLP", [ln2], layer=layer, position=t)
        merged = add_node(
            "ADD",
            [resid[t], attn_out[t], mlp],
            layer=layer, position=t
        )
        new_resid.append(merged)

    resid = new_resid

final_ln = [add_node("FINAL_LN", [resid[t]], position=t) for t in range(T)]
output_node = add_node(
    "UNEMBED",
    [final_ln[-1]],
    position=T-1,
    predicted_token_id=next_id
)

print("Gödel nodes created:", len(nodes))
print("final node:", output_node)
print("formal top-level code:")
print("G_final =", formal_code(output_node))

In [ ]:
# פותחים לאחור את הקוד: מהתוצאה אל המקורות.
def show_backtrace(node_id, depth=0, max_depth=5):
    n = nodes[node_id]
    indent = "  " * depth

    extra = ""
    if n.op == "TOKEN":
        extra = f" token={n.meta.get('token_text')!r} pos={n.meta.get('position')}"
    elif "layer" in n.meta:
        extra = f" L{n.meta.get('layer')}"
        if "head" in n.meta:
            extra += f" H{n.meta.get('head')}"

    print(f"{indent}G{node_id}: {n.op}{extra} [{formal_code(node_id, 3)}]")

    if depth >= max_depth:
        return

    for p in n.parents[:8]:
        show_backtrace(p, depth+1, max_depth)

show_backtrace(output_node)

## מה הוכחנו בסבב הזה?

רק דבר אחד: אפשר לקחת forward pass של מודל pretrained אמיתי ולתאר את היסטוריית החישוב שלו בקוד גדל רקורסיבי בלי לחשב את המספר הענק.

עוד **לא** חיפשנו קטגוריות, gcd או חוקיות בין הרבה prompts. זה השלב הבא, אחרי שנבדוק שה-trace נראה לנו נכון.